In [4]:
import pandas as pd

pd.set_option("display.max_columns", 100)

players = pd.read_csv("C:\\Users\\Nico\\Documents\\The Power MBA\\Proyecto Final\\cs2-pro-scene-analysis\\data\\processed\\players_maps_long.csv", parse_dates=["datetime"])
hltv = pd.read_csv("C:\\Users\\Nico\\Documents\\The Power MBA\\Proyecto Final\\cs2-pro-scene-analysis\\data\\processed\\hltv_players_clean.csv")
print(players.shape, hltv.shape)

(119105, 28) (803, 23)


In [5]:
players["nick_key"] = players["nick"].str.strip().str.lower()
players["country_code"] = players["country"].str.upper()

print("filas sin país:", players["country_code"].isna().sum(),
      "| jugadores:", players.loc[players["country_code"].isna(), "player_id"].nunique())

filas sin país: 9153 | jugadores: 275


In [6]:
cols_hltv = [c for c in hltv.columns if c not in ("nick", "teams", "country_name")]

final = players.merge(
    hltv[cols_hltv],
    on=["nick_key", "country_code"],
    how="left",
    validate="many_to_one",
)
print(final.shape, "| mismas filas que antes:", len(final) == len(players))

(119105, 48) | mismas filas que antes: True


In [7]:
final["has_career_stats"] = final["hltv_player_id"].notna()

print("filas con stats de carrera:", final["has_career_stats"].sum(),
      f"({final['has_career_stats'].mean():.1%})")
print("jugadores con match:", final.loc[final["has_career_stats"], "player_id"].nunique(),
      "de", final["player_id"].nunique())
print(final.groupby("tier")["has_career_stats"].mean().round(3))

filas con stats de carrera: 48965 (41.1%)
jugadores con match: 358 de 1339
tier
1    0.471
2    0.311
3    0.285
Name: has_career_stats, dtype: float64


In [8]:
final = final.drop(columns=["country"])
final["hltv_player_id"] = final["hltv_player_id"].astype("Int64")
final["num_teams"] = final["num_teams"].astype("Int64")

print(final.shape)
assert final.shape[0] >= 50000 and final.shape[1] >= 20, "No cumple los mínimos"
print("duplicados jugador-mapa:", final.duplicated(["game_id", "player_id"]).sum())

(119105, 48)
duplicados jugador-mapa: 0


In [9]:
final.to_csv("C:\\Users\\Nico\\Documents\\The Power MBA\\Proyecto Final\\cs2-pro-scene-analysis\\data\\processed\\cs2_final_dataset.csv", index=False)